In [71]:
import earthaccess
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [2]:
df_mgrs = get_mgrs_table()

In [3]:
earthaccess.login()

# Testing

In [114]:
def get_opera_id(query_item) -> str:
    return dict(query_item.__dict__['render_dict'])['meta']['native-id']

def july_query_dist_hls_cmr(mgrs_tile_id: str, start_time='2025-07-01', stop_time='2025-08-01'):
    collection_short_name = 'OPERA_L3_DIST-ALERT-HLS_V1'
    mgrs_bounds = tuple(df_mgrs[df_mgrs.mgrs_tile_id == mgrs_tile_id].total_bounds)
    mgrs_bounds = tuple(float(x) for x in mgrs_bounds)
    datasets_found = earthaccess.search_data(
        short_name=collection_short_name,
        temporal=(start_time, stop_time),
        cloud_hosted=True,
        bounding_box=mgrs_bounds
    )
    datasets_found = [d for d in datasets_found if mgrs_tile_id in get_opera_id(d)]
    return datasets_found

In [115]:
q = get_dist_hls_df('18SUJ')


(-77.3336328305541, 38.7382095878689, -76.0381048853751, 39.7454935451875)


In [116]:
def get_processing_time(query_item: earthaccess.DataGranule) -> pd.Timestamp:
    data = query_item.__dict__['render_dict']
    ts = pd.Timestamp(data['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'])
    return ts

In [117]:
dict(qs_ordered[0].__dict__['render_dict'])

{'meta': {'concept-type': 'granule',
  'concept-id': 'G3591775663-LPCLOUD',
  'revision-id': 1,
  'native-id': 'OPERA_L3_DIST-ALERT-HLS_T18TUK_20250701T155819Z_20250705T084525Z_S2B_30_v1',
  'collection-concept-id': 'C2746980408-LPCLOUD',
  'provider-id': 'LPCLOUD',
  'format': 'application/vnd.nasa.cmr.umm+json',
  'revision-date': '2025-07-05T08:46:30.122Z'},
 'umm': {'TemporalExtent': {'RangeDateTime': {'BeginningDateTime': '2025-07-01T16:12:08.432590Z',
    'EndingDateTime': '2025-07-01T16:12:08.432590Z'}},
  'GranuleUR': 'OPERA_L3_DIST-ALERT-HLS_T18TUK_20250701T155819Z_20250705T084525Z_S2B_30_v1',
  'AdditionalAttributes': [{'Name': 'Input_DIST-ALERT_granule',
    'Values': ['OPERA_L3_DIST-ALERT-HLS_T18TUK_20250628T154819Z_20250630T123733Z_S2B_30_v1']},
   {'Name': 'BaselineCalendarWindow', 'Values': ['+/- 15 days']},
   {'Name': 'BaselineYearWindow', 'Values': ['3']},
   {'Name': 'BaselineImageIds',
    'Values': ['HLS.S30.T18TUK.2024168T155819.v2.0',
     'HLS.S30.T18TUK.2024170

In [118]:
qs_ordered = sorted(q, key=get_processing_time)

In [119]:
# earthaccess.download(q[0], '.')

# Automate

In [101]:
df_data = pd.read_csv('amy_val_one_of_each__2025_09_15.csv')
names = df_data.name.unique().tolist()
names

['dry_conditions__24MXV',
 'fire__09WXN',
 'high_water_year__29PNP',
 'landslide__18LUR',
 'logging__18MVS',
 'mining__21MWP',
 'new_construction__18SUJ',
 'road_expansion__50SQE',
 'shifting_cultivation__48QXD',
 'tornado__16SGG']

In [120]:
def get_most_recent_july_date(mgrs_tile_id: str, job_name: str) -> str:
    datasets = july_query_dist_hls_cmr(mgrs_tile_id)
    if datasets:
        datasets_ordered = sorted(datasets, key=get_processing_time)
        opera_id = dict(datasets_ordered[-1].__dict__['render_dict'])['meta']['native-id']
        out_dir = Path(f'dist_hls/{job_name}/{opera_id}')
        out_dir.mkdir(exist_ok=True, parents=True)
        r = earthaccess.download(datasets_ordered[-1], out_dir)
        return r
    else:
        return ''

In [121]:
mgrs_tile_ids = [name.split('__')[-1] for name in names]
mgrs_tile_ids[:1]

['24MXV']

In [122]:
_ = [get_most_recent_july_date(m_id, name) for m_id, name in zip(mgrs_tile_ids, tqdm(names[:]))]

  0%|                                                                     | 0/10 [00:00<?, ?it/s]
QUEUEING TASKS | : 100%|███████████████████████████████████████| 19/19 [00:00<00:00, 4879.19it/s]

PROCESSING TASKS | :   0%|                                                | 0/19 [00:00<?, ?it/s]
PROCESSING TASKS | :   5%|██                                      | 1/19 [00:02<00:38,  2.16s/it]
PROCESSING TASKS | :  11%|████▏                                   | 2/19 [00:02<00:20,  1.20s/it]
PROCESSING TASKS | :  16%|██████▎                                 | 3/19 [00:02<00:11,  1.42it/s]
PROCESSING TASKS | :  21%|████████▍                               | 4/19 [00:03<00:10,  1.41it/s]
PROCESSING TASKS | :  26%|██████████▌                             | 5/19 [00:03<00:06,  2.03it/s]
PROCESSING TASKS | :  42%|████████████████▊                       | 8/19 [00:03<00:02,  4.32it/s]
PROCESSING TASKS | :  53%|████████████████████▌                  | 10/19 [00:04<00:01,  5.46it/s]
PROCESSING TASKS | 

[['dist_hls/dry_conditions__24MXV/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1_VEG-DIST-STATUS.tif',
  'dist_hls/dry_conditions__24MXV/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1_VEG-IND.tif',
  'dist_hls/dry_conditions__24MXV/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1_VEG-ANOM.tif',
  'dist_hls/dry_conditions__24MXV/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1_VEG-HIST.tif',
  'dist_hls/dry_conditions__24MXV/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1/OPERA_L3_DIST-ALERT-HLS_T24MXV_20250731T130249Z_20250813T070002Z_S2B_30_v1_VEG-ANOM-MAX.t